# SPY Volatility Regime Detection — IOHMM Analysis

**Training period:** 2019-01-01 to 2023-12-31  
**Test period:** 2024-01-01 to 2024-12-31  

This notebook fits a 3-state Gaussian Input-Output HMM on SPY log-realised volatility using cross-asset features (TLT, HYG, UUP, GLD), then evaluates regime identification quality in- and out-of-sample.

In [ ]:
import sys, os

# Ensure project root is on the path so absolute imports work
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", "..", ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.colors import ListedColormap

from IOHMM.regimes.features import build_vol_iohmm_dataset
from IOHMM.regimes.iohmm import GaussianIOHMM
from IOHMM.regimes.diagnostics import summarize_regimes
from data_preprocessing.data_adapter import YFinanceAdapter

warnings.filterwarnings("ignore", category=UserWarning)

plt.rcParams.update({
    "figure.figsize": (14, 5),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})

TRAIN_START = "2019-01-01"
TRAIN_END   = "2023-12-31"
TEST_START  = "2024-01-01"
TEST_END    = "2024-12-31"

REGIME_COLORS = ["#2ecc71", "#f39c12", "#e74c3c"]  # low / mid / high vol
REGIME_LABELS = ["Low Vol", "Mid Vol", "High Vol"]

print("imports ok")

## 1. Data Loading & Feature Preparation

In [ ]:
adapter = YFinanceAdapter()
tickers = ["SPY", "TLT", "HYG", "UUP", "GLD"]

raw = adapter.get_data(
    tickers=tickers,
    start_date="2018-06-01",   # lead time for rolling windows
    end_date="2025-01-31",
    force_refresh=False,
)

prepared = build_vol_iohmm_dataset(
    raw,
    target_ticker="SPY",
    external_tickers=("TLT", "HYG", "UUP", "GLD"),
    rv_window_target=10,
    rv_window_external=5,
    strictly_external_inputs=True,
)

dates = prepared.dates
train_mask = (dates >= TRAIN_START) & (dates <= TRAIN_END)
test_mask  = (dates >= TEST_START)  & (dates <= TEST_END)

X_train, y_train, dates_train = prepared.X[train_mask], prepared.y[train_mask], dates[train_mask]
X_test,  y_test,  dates_test  = prepared.X[test_mask],  prepared.y[test_mask],  dates[test_mask]

print(f"Train: {dates_train[0].date()} -> {dates_train[-1].date()}  ({len(y_train)} obs)")
print(f"Test:  {dates_test[0].date()} -> {dates_test[-1].date()}  ({len(y_test)} obs)")
print(f"Features ({len(prepared.feature_names)}):")
for f in prepared.feature_names:
    print(f"  {f}")

## 2. Model Fitting (Train Period Only)

In [ ]:
model = GaussianIOHMM(
    n_states=3,
    emission_ridge=1e-4,
    transition_l2=1e-3,
    max_iter=150,
    tol=1e-4,
)

fit_result = model.fit(X_train, y_train)

print(f"Converged: {fit_result.converged}")
print(f"Iterations: {fit_result.n_iter}")
print(f"Final train log-lik: {fit_result.log_likelihoods[-1]:.4f}")
print(f"Test log-lik:        {model.score(X_test, y_test):.4f}")

## 3. EM Convergence Diagnostic

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

ll = fit_result.log_likelihoods

# Log-likelihood trajectory
axes[0].plot(ll, "o-", markersize=3, color="steelblue")
axes[0].set_xlabel("EM Iteration")
axes[0].set_ylabel("Log-Likelihood")
axes[0].set_title("EM Log-Likelihood Trajectory")

# Iteration-over-iteration improvement
improvements = np.diff(ll)
axes[1].bar(range(1, len(ll)), improvements, color="steelblue", alpha=0.7)
axes[1].axhline(0, color="k", linewidth=0.5)
axes[1].set_xlabel("EM Iteration")
axes[1].set_ylabel("LL Improvement")
axes[1].set_title("Per-Iteration LL Improvement")

plt.tight_layout()
plt.show()

# Sanity check: LL should be monotonically non-decreasing
decreases = np.sum(improvements < -1e-6)
print(f"LL decreases (> 1e-6): {decreases}  {'OK' if decreases == 0 else 'WARNING'}")

## 4. Regime Inference (Train & Test)

In [ ]:
train_results = model.make_results_frame(
    dates=dates_train, X=X_train, y=y_train, state_labels=REGIME_LABELS, use_viterbi=True,
)
test_results = model.make_results_frame(
    dates=dates_test, X=X_test, y=y_test, state_labels=REGIME_LABELS, use_viterbi=True,
)

# Re-label states so that state 0 = lowest mean y (Low Vol)
state_means = train_results.groupby("state")["y"].mean().sort_values()
label_map = {old: new for new, old in enumerate(state_means.index)}
for df in [train_results, test_results]:
    df["state"] = df["state"].map(label_map)

print("=== TRAIN regime summary ===")
print(summarize_regimes(train_results))
print()
print("=== TEST regime summary ===")
print(summarize_regimes(test_results))

## 5. In-Sample Regime Overlay (2019-2023)

Log-realised volatility coloured by Viterbi-decoded regime, with regime-shaded background.

In [ ]:
def plot_regime_timeseries(results, title, regime_colors, regime_labels):
    """Plot y with regime-shaded background."""
    fig, ax = plt.subplots(figsize=(16, 5))

    dates = results.index
    y = results["y"].values
    states = results["state"].values

    ax.plot(dates, y, color="k", linewidth=0.6, alpha=0.8, label="log RV")

    # Shade background by regime
    prev_state = states[0]
    start_idx = 0
    for i in range(1, len(states)):
        if states[i] != prev_state or i == len(states) - 1:
            end_idx = i if states[i] != prev_state else i + 1
            ax.axvspan(
                dates[start_idx], dates[min(end_idx, len(dates) - 1)],
                alpha=0.2, color=regime_colors[prev_state], linewidth=0,
            )
            start_idx = i
            prev_state = states[i]

    # Legend patches
    from matplotlib.patches import Patch
    handles = [Patch(facecolor=c, alpha=0.3, label=l) for c, l in zip(regime_colors, regime_labels)]
    ax.legend(handles=handles, loc="upper right", framealpha=0.9)

    ax.set_title(title)
    ax.set_ylabel("Log Realised Volatility")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()


plot_regime_timeseries(train_results, "In-Sample Regime Detection (2019-2023)", REGIME_COLORS, REGIME_LABELS)

## 6. Out-of-Sample Regime Overlay (2024)

In [ ]:
plot_regime_timeseries(test_results, "Out-of-Sample Regime Detection (2024)", REGIME_COLORS, REGIME_LABELS)

## 7. State Posterior Probabilities

Stacked area plot of filtered state probabilities. Sharp transitions = confident regime identification; blurred regions = model uncertainty.

In [ ]:
def plot_state_posteriors(results, title, regime_colors, regime_labels, label_map):
    """Stacked area chart of state posterior probabilities."""
    fig, ax = plt.subplots(figsize=(16, 4))

    dates = results.index
    # Get probability columns in re-labelled order
    prob_cols_orig = [c for c in results.columns if c.startswith("p_")]
    # Build probabilities in sorted (relabelled) state order
    probs = np.zeros((len(results), len(regime_labels)))
    for orig_state, new_state in label_map.items():
        probs[:, new_state] = results[prob_cols_orig[orig_state]].values

    ax.stackplot(
        dates, probs.T,
        labels=regime_labels,
        colors=regime_colors,
        alpha=0.7,
    )
    ax.set_ylim(0, 1)
    ax.set_ylabel("State Probability")
    ax.set_title(title)
    ax.legend(loc="upper right", framealpha=0.9)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()


plot_state_posteriors(train_results, "State Posteriors — Train (2019-2023)", REGIME_COLORS, REGIME_LABELS, label_map)
plot_state_posteriors(test_results,  "State Posteriors — Test (2024)",       REGIME_COLORS, REGIME_LABELS, label_map)

## 8. Emission Distribution by Regime

Histograms of log-RV within each regime. Well-separated distributions with low overlap = strong regime discrimination.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)

for ax, (results, period) in zip(axes, [(train_results, "Train"), (test_results, "Test")]):
    for state in sorted(results["state"].unique()):
        vals = results.loc[results["state"] == state, "y"]
        ax.hist(vals, bins=30, alpha=0.5, color=REGIME_COLORS[state],
                label=f"{REGIME_LABELS[state]} (n={len(vals)})", density=True)
    ax.set_xlabel("Log Realised Volatility")
    ax.set_ylabel("Density")
    ax.set_title(f"Emission Distributions — {period}")
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

## 9. Regime Duration Analysis

Distribution of consecutive days spent in each regime. Persistent regimes (long runs) are a sign of meaningful latent structure rather than noise-fitting.

In [ ]:
from IOHMM.regimes.diagnostics import _state_run_lengths

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)

for k in range(3):
    ax = axes[k]
    train_runs = _state_run_lengths(train_results["state"].values, k)
    test_runs  = _state_run_lengths(test_results["state"].values, k)

    if train_runs:
        ax.hist(train_runs, bins=range(1, max(train_runs) + 2), alpha=0.6,
                color=REGIME_COLORS[k], label=f"Train (n={len(train_runs)})")
    if test_runs:
        ax.hist(test_runs, bins=range(1, max(test_runs) + 2), alpha=0.6,
                color=REGIME_COLORS[k], edgecolor="k", linestyle="--",
                label=f"Test (n={len(test_runs)})")

    ax.set_title(f"{REGIME_LABELS[k]} Duration")
    ax.set_xlabel("Days")
    ax.legend(fontsize=9)

axes[0].set_ylabel("Count")
plt.suptitle("Regime Duration Distributions", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 10. Transition Matrix Heatmap

Empirical transition counts from the Viterbi path (train set). Diagonal-dominant = sticky regimes.

In [ ]:
def empirical_transition_matrix(states, n_states):
    """Count-based transition matrix from a state sequence."""
    counts = np.zeros((n_states, n_states))
    for s_prev, s_next in zip(states[:-1], states[1:]):
        counts[s_prev, s_next] += 1
    row_sums = counts.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1
    return counts / row_sums


fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (results, period) in zip(axes, [(train_results, "Train"), (test_results, "Test")]):
    T = empirical_transition_matrix(results["state"].values, 3)
    im = ax.imshow(T, cmap="YlOrRd", vmin=0, vmax=1)
    for i in range(3):
        for j in range(3):
            ax.text(j, i, f"{T[i, j]:.2f}", ha="center", va="center", fontsize=12,
                    color="white" if T[i, j] > 0.6 else "black")
    ax.set_xticks(range(3))
    ax.set_yticks(range(3))
    ax.set_xticklabels(REGIME_LABELS, fontsize=10)
    ax.set_yticklabels(REGIME_LABELS, fontsize=10)
    ax.set_xlabel("To")
    ax.set_ylabel("From")
    ax.set_title(f"Empirical Transitions — {period}")

plt.colorbar(im, ax=axes, shrink=0.8, label="P(transition)")
plt.tight_layout()
plt.show()

## 11. Emission Model Parameters

Learned intercepts (regime-level mean), variance, and feature betas for each state.

In [ ]:
em = model.emission_model
beta_names = ["intercept"] + prepared.feature_names

rows = []
for k, st in enumerate(em.states):
    mapped_k = label_map[k]
    row = {"regime": REGIME_LABELS[mapped_k], "sigma": np.sqrt(st.sigma2)}
    for name, val in zip(beta_names, st.beta):
        row[name] = round(val, 4)
    rows.append(row)

param_df = pd.DataFrame(rows).set_index("regime").sort_index()
print("Emission parameters (standardised feature space):")
param_df

## 12. Model Health Checks

A set of quantitative sanity checks that flag potential issues.

In [ ]:
checks = []

# 1. Convergence
checks.append(("EM converged", fit_result.converged, ""))

# 2. No LL decreases
ll_diffs = np.diff(fit_result.log_likelihoods)
n_decreases = np.sum(ll_diffs < -1e-6)
checks.append(("No LL decreases", n_decreases == 0, f"{n_decreases} decreases"))

# 3. All regimes populated in train
train_counts = train_results["state"].value_counts()
all_populated = len(train_counts) == 3 and train_counts.min() >= 10
checks.append(("All 3 regimes populated (train)", all_populated,
               f"counts: {dict(train_counts.sort_index())}"))

# 4. All regimes populated in test
test_counts = test_results["state"].value_counts()
checks.append(("All 3 regimes populated (test)", len(test_counts) >= 2,
               f"counts: {dict(test_counts.sort_index())}"))

# 5. Emission means are ordered (low < mid < high)
train_means = train_results.groupby("state")["y"].mean().sort_index()
ordered = train_means.is_monotonic_increasing
checks.append(("Emission means ordered (low < mid < high)", ordered,
               f"means: {[f'{v:.3f}' for v in train_means.values]}"))

# 6. Test log-lik per observation not catastrophically worse than train
train_ll_per_obs = model.score(X_train, y_train) / len(y_train)
test_ll_per_obs  = model.score(X_test, y_test) / len(y_test)
ll_ratio = test_ll_per_obs / train_ll_per_obs if train_ll_per_obs != 0 else 0
ll_ok = ll_ratio > 0.5  # test per-obs LL is at least 50% of train
checks.append(("Test LL/obs not catastrophically worse",  ll_ok,
               f"train={train_ll_per_obs:.3f}  test={test_ll_per_obs:.3f}"))

# 7. Transition matrix is diagonal-dominant (sticky)
T_emp = empirical_transition_matrix(train_results["state"].values, 3)
diag_dominant = all(T_emp[i, i] > 0.5 for i in range(3) if T_emp[i].sum() > 0)
checks.append(("Transition matrix diagonal-dominant", diag_dominant,
               f"diag: {[f'{T_emp[i,i]:.2f}' for i in range(3)]}"))

# 8. Variance separation
sigmas = sorted([np.sqrt(st.sigma2) for st in model.emission_model.states])
var_separated = sigmas[-1] / sigmas[0] > 1.2 if sigmas[0] > 0 else False
checks.append(("Variance separation across states", var_separated,
               f"sigmas: {[f'{s:.4f}' for s in sigmas]}"))

print(f"{'Check':<50} {'Pass':>6}   Detail")
print("-" * 90)
for name, passed, detail in checks:
    status = "PASS" if passed else "FAIL"
    print(f"{name:<50} {status:>6}   {detail}")

## 13. Regime Overlay on SPY Price (2024 OOS)

Map regimes back onto the SPY close price to see if high-vol regimes align with drawdowns.

In [ ]:
# Get SPY close prices for both periods
spy_close = raw[("Close", "SPY")].dropna()

fig, axes = plt.subplots(2, 1, figsize=(16, 9), sharex=False)

for ax, (results, period) in zip(axes, [(train_results, "Train 2019-2023"),
                                         (test_results, "Test 2024")]):
    price = spy_close.reindex(results.index).dropna()
    common = results.index.intersection(price.index)
    price = price.loc[common]
    states = results.loc[common, "state"].values

    ax.plot(common, price.values, color="k", linewidth=0.8)

    prev_state = states[0]
    start_idx = 0
    for i in range(1, len(states)):
        if states[i] != prev_state or i == len(states) - 1:
            end_idx = i if states[i] != prev_state else i + 1
            ax.axvspan(
                common[start_idx], common[min(end_idx, len(common) - 1)],
                alpha=0.2, color=REGIME_COLORS[prev_state], linewidth=0,
            )
            start_idx = i
            prev_state = states[i]

    ax.set_ylabel("SPY Close ($)")
    ax.set_title(f"SPY Price with Regime Overlay — {period}")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))

from matplotlib.patches import Patch
fig.legend(
    handles=[Patch(facecolor=c, alpha=0.3, label=l) for c, l in zip(REGIME_COLORS, REGIME_LABELS)],
    loc="upper center", ncol=3, framealpha=0.9, bbox_to_anchor=(0.5, 1.0),
)
fig.autofmt_xdate()
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

## 14. Feature Importance by Regime

Absolute beta coefficients per regime — shows which cross-asset signals drive each state's emission model.

In [ ]:
feature_names = prepared.feature_names
n_feat = len(feature_names)

fig, ax = plt.subplots(figsize=(12, max(4, n_feat * 0.35)))

bar_width = 0.25
y_pos = np.arange(n_feat)

for k, st in enumerate(em.states):
    mapped_k = label_map[k]
    # skip intercept (index 0), take feature betas
    betas = np.abs(st.beta[1:])
    ax.barh(y_pos + mapped_k * bar_width, betas, bar_width,
            color=REGIME_COLORS[mapped_k], alpha=0.7, label=REGIME_LABELS[mapped_k])

ax.set_yticks(y_pos + bar_width)
ax.set_yticklabels(feature_names, fontsize=9)
ax.set_xlabel("|Beta| (standardised)")
ax.set_title("Feature Importance by Regime (|emission betas|)")
ax.legend()
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 15. Save Results

In [ ]:
train_results.to_csv("spy_vol_iohmm_train_results.csv")
test_results.to_csv("spy_vol_iohmm_test_results.csv")
summarize_regimes(train_results).to_csv("spy_vol_iohmm_train_summary.csv", index=False)
summarize_regimes(test_results).to_csv("spy_vol_iohmm_test_summary.csv", index=False)

print("Saved: spy_vol_iohmm_{train,test}_{results,summary}.csv")